**PySpark Session Setup**

In [3]:
# install PySpark
! pip install pyspark >& /dev/null

#start session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("wheat_futures_price_prediction").getOrCreate()

#upload libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

25/04/19 18:53:30 WARN Utils: Your hostname, codespaces-66c291 resolves to a loopback address: 127.0.0.1; using 10.0.2.151 instead (on interface eth0)
25/04/19 18:53:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/19 18:53:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
#Read weather and pricing data
weather_raw = spark.read.csv('weather_data_RAW.csv', header=True, inferSchema=True)
wheat_price_data = spark.read.csv('US_wheat_future_pricing_RAW.csv', header=True, inferSchema=True)

#Parse date columns
from pyspark.sql.functions import to_date, col

weather = weather_raw.withColumn(
    "Date",
    to_date(col("datetime"), "M/d/yyyy")      
).drop("datetime")                            

pricing = wheat_price_data.withColumn(
    "Date",
    to_date(col("Date"), "M/d/yyyy")        
)

#Join on Date
merged = weather.join(pricing, on="Date", how="left")

merged.show(10, truncate=False)


25/04/19 18:53:44 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
25/04/19 18:53:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+-----+-------+-------+----+------------+------------+---------+----+--------+------+----------+-----------+----------+----+---------+--------+---------+-------+----------------+----------+----------+--------------+-----------+-------+----------+-------------------+-------------------+---------+----------------------------+--------------------------------------------------------------------------------+-----------------+----------------------------------------------------------+------+------+------+------+------+--------+
|Date      |name |tempmax|tempmin|temp|feelslikemax|feelslikemin|feelslike|dew |humidity|precip|precipprob|precipcover|preciptype|snow|snowdepth|windgust|windspeed|winddir|sealevelpressure|cloudcover|visibility|solarradiation|solarenergy|uvindex|severerisk|sunrise            |sunset             |moonphase|conditions                  |description                                                                     |icon             |stations                  